# GCS navigation and pickup

In [1]:
import math
from pathlib import Path

import plotly.graph_objects as go

import coraplex
from coraplex.datastructures.dataclasses import Context
from coraplex.datastructures.enums import Arms, ApproachDirection, VerticalAlignment
from coraplex.datastructures.grasp import GraspDescription
from coraplex.execution_environment import simulated_robot
from coraplex.plans.factories import sequential
from coraplex.robot_plans.actions.core.navigation import NavigateAction
from coraplex.robot_plans.actions.core.pick_up import PickUpAction
from coraplex.robot_plans.actions.core.robot_body import ParkArmsAction
from coraplex.view_manager import ViewManager
from semantic_digital_twin.adapters.mesh import STLParser
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix, Point3, Pose
from semantic_digital_twin.world_description.connections import Connection6DoF, FixedConnection, OmniDrive
from semantic_digital_twin.world_description.geometry import BoundingBox, Box, Color, Scale
from semantic_digital_twin.world_description.graph_of_convex_sets import GraphOfConvexSets
from semantic_digital_twin.world_description.shape_collection import BoundingBoxCollection, ShapeCollection
from semantic_digital_twin.world_description.world_entity import Body

In [2]:
world = URDFParser.from_file(PR2.get_ros_file_path()).parse()
PR2.from_world(world)

with world.modify_world():
    robot_root = world.root
    map_body = Body(name=PrefixedName("map"))
    localization_body = Body(name=PrefixedName("odom_combined"))
    world.add_connection(
        Connection6DoF.create_with_dofs(world, map_body, localization_body)
    )
    drive = OmniDrive.create_with_dofs(
        parent=localization_body,
        child=robot_root,
        world=world,
    )
    world.add_connection(drive)
    drive.has_hardware_interface = True

drive.origin = HomogeneousTransformationMatrix.from_xyz_rpy(x=2.0)

with world.modify_world():
    route_obstacle = Body(
        name=PrefixedName("route_obstacle"),
        collision=ShapeCollection([Box(scale=Scale(1.0, 1.0, 0.95))]),
        visual=ShapeCollection(
            [Box(scale=Scale(1.0, 1.0, 0.95), color=Color(0.8, 0.2, 0.2, 1.0))]
        ),
    )
    pickup_surface = Body(
        name=PrefixedName("pickup_surface"),
        collision=ShapeCollection([Box(scale=Scale(1.0, 1.0, 0.95))]),
        visual=ShapeCollection(
            [Box(scale=Scale(1.0, 1.0, 0.95), color=Color(0.4, 0.4, 0.4, 1.0))]
        ),
    )
    world.add_connection(
        FixedConnection(
            parent=world.root,
            child=route_obstacle,
            parent_T_connection_expression=HomogeneousTransformationMatrix.from_xyz_rpy(
                x=0.5, z=0.475, reference_frame=world.root
            ),
        )
    )
    world.add_connection(
        FixedConnection(
            parent=world.root,
            child=pickup_surface,
            parent_T_connection_expression=HomogeneousTransformationMatrix.from_xyz_rpy(
                x=-1.8, z=0.475, reference_frame=world.root
            ),
        )
    )

coraplex_root = Path(coraplex.__file__).resolve().parents[2]
milk_world = STLParser(str(coraplex_root / "resources" / "objects" / "milk.stl")).parse()
world.merge_world_at_pose(
    milk_world,
    HomogeneousTransformationMatrix.from_xyz_rpy(
        x=-1.5,
        z=1.02,
        yaw=math.pi,
        reference_frame=world.root,
    ),
)

robot = world.get_semantic_annotations_by_type(PR2)[0]
milk = world.get_body_by_name("milk.stl")
context = Context(world=world, robot=robot)

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]


In [3]:
# ROS setup
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [4]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

ROS2 publishers started


In [5]:
search_space = BoundingBoxCollection(
    [
        BoundingBox(
            min_x=-1.0,
            max_x=2.5,
            min_y=-1.4,
            max_y=1.4,
            min_z=0.0,
            max_z=2.0,
            origin=HomogeneousTransformationMatrix(reference_frame=world.root),
        )
    ],
    reference_frame=world.root,
)

graph_of_convex_sets = GraphOfConvexSets.navigation_map_from_world(
    world=world,
    search_space=search_space,
    bloat_obstacles=0.15,
)
graph_of_convex_sets.create_as_region(name=PrefixedName("navigation_free_space"))

start = Point3(2.0, 0.0, 0.0, reference_frame=world.root)
pickup_position = Point3(-0.8, -0.4, 0.0, reference_frame=world.root)
path = graph_of_convex_sets.path_from_to(start, pickup_position)
assert path is not None and len(path) > 2
[(round(float(point.x), 3), round(float(point.y), 3)) for point in path]

[(2.0, 0.0), (1.15, 1.025), (-0.15, 1.025), (-0.8, -0.4)]

In [6]:
figure = go.Figure(graph_of_convex_sets.plot_free_space())
figure.add_trace(
    go.Scatter3d(
        x=[point.x for point in path],
        y=[point.y for point in path],
        z=[0.05 for point in path],
        mode="lines+markers",
        name="GCS path",
        line={"color": "black", "width": 8},
    )
)
figure.update_layout(height=550, scene_aspectmode="data")
figure

In [7]:
import time
from functools import partial

from coraplex.plans.factories import code, sequential


# navigation_poses = [
#     Pose.from_xyz_rpy(
#         x=point.x,
#         y=point.y,
#         z=0.0,
#         yaw=0.0,
#         reference_frame=world.root,
#     )
#     for point in path[1:-1]
# ]
# pickup_yaw = math.atan2(
#     float(milk.global_pose.y - pickup_position.y),
#     float(milk.global_pose.x - pickup_position.x),
# )
# navigation_poses.append(
#     Pose.from_xyz_rpy(
#         x=pickup_position.x,
#         y=pickup_position.y,
#         z=0.0,
#         yaw=pickup_yaw,
#         reference_frame=world.root,
#     )
# )

navigation_step_distance = 0.05
seconds_per_navigation_step = 0.3
navigation_poses = []

for segment_start, segment_end in zip(path, path[1:]):
  start_x = float(segment_start.x)
  start_y = float(segment_start.y)
  distance_x = float(segment_end.x) - start_x
  distance_y = float(segment_end.y) - start_y

  segment_distance = math.hypot(distance_x, distance_y)
  segment_step_count = max(
      1,
      math.ceil(segment_distance / navigation_step_distance),
  )
  segment_yaw = math.atan2(distance_y, distance_x)

  for segment_step_index in range(1, segment_step_count + 1):
      progress = segment_step_index / segment_step_count
      navigation_poses.append(
          Pose.from_xyz_rpy(
              x=start_x + distance_x * progress,
              y=start_y + distance_y * progress,
              z=0.0,
              yaw=segment_yaw,
              reference_frame=world.root,
          )
      )

pickup_yaw = math.atan2(
  float(milk.global_pose.y - pickup_position.y),
  float(milk.global_pose.x - pickup_position.x),
)
navigation_poses[-1] = Pose.from_xyz_rpy(
  x=pickup_position.x,
  y=pickup_position.y,
  z=0.0,
  yaw=pickup_yaw,
  reference_frame=world.root,
)


grasp_description = GraspDescription(
    approach_direction=ApproachDirection.FRONT,
    vertical_alignment=VerticalAlignment.NoAlignment,
    end_effector=ViewManager.get_end_effector_view(Arms.LEFT, robot),
)

navigation_steps = []

for navigation_pose in navigation_poses:
  navigation_steps.extend(
      [
          NavigateAction(navigation_pose),
          code(partial(time.sleep, seconds_per_navigation_step))
      ]
  )

task = sequential(
  [
      ParkArmsAction(Arms.BOTH),
      *navigation_steps,
      PickUpAction(milk, Arms.LEFT, grasp_description),
  ],
  context=context,
)

#
# task = sequential(
#     [
#         ParkArmsAction(Arms.BOTH),
#         *[NavigateAction(pose) for pose in navigation_poses],
#         PickUpAction(milk, Arms.LEFT, grasp_description),
#     ],
#     context=context,
# )

with simulated_robot:
    task.perform()

In [ ]:
picked_up = world.get_connection(
    ViewManager.get_end_effector_view(Arms.LEFT, robot).tool_frame,
    milk,
) is not None
assert picked_up

{
    "graph_nodes": len(graph_of_convex_sets.graph.nodes()),
    "graph_edges": len(graph_of_convex_sets.graph.edges()),
    "path_waypoints": len(path),
    "picked_up": picked_up,
}